# W35–W38 · Capstone 毕业项目指南

> 12 个月学习工程的收官之作。三选一，4 周开发（W35–W38）+ 2 周答辩材料（W39–W40）。
> 本 notebook 是**项目规范**：选题、里程碑、评估指标、验收标准、报告模板。

## 学习目标

1. 从三个选题中确定一个，并写出可执行的项目计划；
2. 为项目定义**可量化**的评估指标与验收标准（含置信区间）；
3. 按周推进里程碑，用「最小可行 demo → 迭代加固」的节奏控制风险；
4. 产出符合工程规范的交付物：代码、实验报告、演示视频、复现指南。

## 通用要求（三个选题都必须满足）

- **代码**：放在本仓库 `src/` 或独立仓库，遵守 AGENTS.md 工程规范（uv、pytest、ruff）；
- **实验**：所有结论必须可复现——固定 seed、记录依赖版本、训练产物入 `runs/`；
- **报告**：按第 6 节模板撰写，失败实验与成功实验同等记录；
- **演示**：3–5 分钟视频（W39–W40 制作），含仿真画面 + 指标曲线 + 你的讲解音轨或字幕。

## 1. 选题一：无人机穿越（Drone Gate Racing）

**一句话**：训练四旋翼以 ≥2 m/s 穿越障碍门序列，加域随机化，最后导出 ROS2 策略节点
在 Gazebo 中复现（打通阶段三）。

**技术栈**：Isaac Lab（或阶段二的 MotrixLab）训练 → ONNX 导出 → ROS2 Jazzy 策略节点 → Gazebo 验证。

### 里程碑拆解（W35–W38）

| 周 | 里程碑 | 验收物 |
|----|--------|--------|
| W35 | 环境搭建：USD 场地 + 门资产；Direct 任务骨架；随机策略可运行 | `list_envs` 可见自定义任务；随机 rollout 无 NaN |
| W36 | PPO 基线：单门穿越成功；课程学习（单门→3 门→5 门） | TensorBoard 曲线 + 单门成功率 ≥80% (n≥100) |
| W37 | 域随机化（质量/推力/风扰/观测噪声）+ 鲁棒性评估 | DR 开启前后成功率对比表 |
| W38 | ONNX 导出 → ROS2 节点 → Gazebo 复现 + 指标复测 | Gazebo 录屏 + Sim2Sim 差距分析 |

### 评估指标

| 指标 | 定义 | 目标线 | 优秀线 |
|------|------|--------|--------|
| 穿越成功率 | 5 门连续穿越无碰撞 / n=100 trials | ≥ 60% | ≥ 85% |
| 平均用时 | 成功 trials 的完成时间均值 | ≤ 30 s | ≤ 20 s |
| 碰撞率 | 碰撞 trials 占比 | ≤ 20% | ≤ 5% |
| Sim2Sim 保持率 | Gazebo 成功率 / Isaac 成功率 | ≥ 70% | ≥ 90% |

**风险与对策**：姿态环不稳 → 降低控制频率 + 动作平滑惩罚；门检测难 → 先用真值位姿，
视觉作为加分项；Gazebo 动力学差距 → 加大 DR 范围而非调参硬扛。

## 2. 选题二：四足导航（Quadruped Navigation）

**一句话**：MotrixLab/Isaac Lab 训速度跟踪行走策略（低层），Nav2 做路径规划（高层），
打通「导航 → 运控」分层架构——这是工业界四足产品的标准形态。

**技术栈**：阶段二四足经验 + 阶段三 Nav2/ros2_control + 策略节点封装。

### 里程碑拆解

| 周 | 里程碑 | 验收物 |
|----|--------|--------|
| W35 | 行走策略：速度指令跟踪（平坦地形）达标 | $v_x$ 跟踪误差 ≤ 0.1 m/s；连续行走 60 s 不跌倒 |
| W36 | 地形加固：随机起伏 + 摩擦随机化 + 推搡扰动 | 扰动下跌倒率 ≤ 5% (n=100) |
| W37 | 分层集成：Nav2 发布速度指令 → 策略节点 → 关节力矩 | 仿真中从 A 到 B 自主导航成功 |
| W38 | 全链路评估 + 报告 | 导航成功率统计 + 分层架构时延分析 |

### 评估指标

| 指标 | 定义 | 目标线 | 优秀线 |
|------|------|--------|--------|
| 速度跟踪 RMSE | cmd vs 实际（vx, vy, wz 三通道） | ≤ 0.15 | ≤ 0.08 |
| 跌倒率 | 扰动测试集上翻倒占比 | ≤ 10% | ≤ 3% |
| 导航成功率 | 到达目标点且无碰撞 / n=50 | ≥ 70% | ≥ 90% |
| 单位距离能耗 | 关节功 / 米（相对量纲即可） | 基线 ≤ +30% | ≤ 基线 |

**风险与对策**：Nav2 默认面向轮式 → 把四足抽象成「全向轮底座」，足端细节全交低层；
指令突变导致跌倒 → 在策略观测中加入指令历史，或高层加加速度限幅。

## 3. 选题三：抓取模仿学习（Manipulation IL）

**一句话**：采集示教数据（脚本或遥操作），用 BC/ACT 训练机械臂抓取策略，
在 Isaac Lab 中评估成功率——体验「数据驱动」与 RL 的不同工作流。

**技术栈**：Isaac Lab 遥操作/脚本示教 → HDF5/LeRobot 数据集 → BC（imitation 库，阶段一经验）
或 ACT（transformer 策略）→ 成功率评估。

### 里程碑拆解

| 周 | 里程碑 | 验收物 |
|----|--------|--------|
| W35 | 数据管线：脚本示教（RRT/逆运动学）或遥操作，采 100+ 条成功演示 | 数据集 + 数据集卡（字段/统计/采集协议） |
| W36 | BC 基线：状态观测（关节+目标位姿）→ 动作 | BC 成功率 ≥ 40% (n=100) |
| W37 | ACT（或加图像观测）：对比 BC，分析协变量偏移 | ACT vs BC 对比表 + 失败 case 分类 |
| W38 | 鲁棒性：初始位姿扰动下的成功率曲线 + 报告 | 扰动-成功率曲线 + 数据量 scaling 实验 |

### 评估指标

| 指标 | 定义 | 目标线 | 优秀线 |
|------|------|--------|--------|
| 抓取成功率 | 抓起目标并抬升 5 cm / n=100，**报 Wilson 95% CI** | ≥ 50% | ≥ 80% |
| 数据效率 | 成功率 vs 演示条数曲线（25/50/100/200） | 单调上升 | 100 条即 ≥70% |
| 推理延迟 | 单步前向耗时（部署机器） | ≤ 50 ms | ≤ 20 ms |
| 失败可归因率 | 失败 case 被人工分类覆盖的比例 | 100% | — |

**风险与对策**：遥操作门槛高 → 先用脚本示教保底；BC 协变量偏移 → DAgger 或加噪声增广；
ACT 训练慢 → 减小 chunk size 或换更小 backbone。

## 4. 通用验收 Rubric（满分 100，70 合格）

| 维度 | 权重 | 合格（拿满 60%） | 优秀（拿满） |
|------|------|------------------|--------------|
| 可复现性 | 25 | README 复现指南 + 固定 seed + 依赖锁定 | 一键脚本复现主要结果，含环境 dockerfile/lock |
| 实验严谨性 | 30 | 指标有 n 和波动范围；有 ≥1 组消融 | 置信区间 + 多 seed + 失败 case 分析 |
| 工程质量 | 20 | 代码分层清晰、通过 ruff/pytest | 有 CI、配置化超参、模块可单测 |
| 报告与表达 | 25 | 按模板完整、图表规范 | 有洞察的失败分析 + 清晰的架构图 |

**硬性否决项**：抄袭、伪造实验数据、报告中的曲线无法由仓库代码复现。

## 5. 评估的统计学：成功率必须带置信区间

「跑了 20 次成功 18 次，成功率 90%」在统计上几乎不可信——样本太少。
二项分布的成功率估计要用置信区间说话。Wilson 区间（小样本也稳健）：

$$\text{CI} = \frac{1}{1+\frac{z^2}{n}}\left(\hat{p} + \frac{z^2}{2n} \pm z\sqrt{\frac{\hat{p}(1-\hat{p})}{n} + \frac{z^2}{4n^2}}\right)$$

下面这个 cell 可在本机直接运行，体会 n 对结论可信度的影响：

In [ ]:
# 运行前提：仅标准库。本 cell 可在本机运行。
import math


def wilson_ci(k: int, n: int, z: float = 1.96) -> tuple[float, float]:
    """二项成功率的 Wilson 置信区间。k=成功次数, n=总次数, z=1.96 对应 95%。"""
    if n == 0:
        raise ValueError("n 必须大于 0")
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return center - margin, center + margin


print(f"{'观测':>10} {'点估计':>8} {'95% Wilson CI':>22} {'区间宽度':>8}")
for n in (20, 50, 100, 400):
    k = round(0.85 * n)
    lo, hi = wilson_ci(k, n)
    print(f"{k:>4}/{n:<5} {k/n:>7.0%}   [{lo:.3f}, {hi:.3f}] {hi-lo:>12.3f}")

print()
print("要点：同样 85% 的点估计，n=20 时真值可能在 [0.64, 0.95] 之间；")
print("n=400 时才能说成功率大概率不低于 0.81。Capstone 验收以 n=100 为底线。")

## 6. 报告模板（`docs/capstone_report.md`，建议 3000–5000 字 + 图表）

```
# <项目名> Capstone 报告

## 0. 摘要（200 字：做了什么、关键数字、最大教训）
## 1. 问题定义
   - 任务描述、MDP 五元组、成功判据（引用验收指标表）
## 2. 方法
   - 架构图（仿真/算法/部署三层）、关键设计决策与理由
## 3. 实验设置
   - 硬件、软件版本（pip freeze / git commit）、超参表、随机种子
## 4. 结果
   - 主指标表（含 CI/n）、学习曲线、行为视频链接
## 5. 消融与分析
   - 至少 1 组消融；失败 case 分类统计；Sim2Sim/Sim2Real 差距分析
## 6. 结论与局限
   - 诚实的局限清单（比吹嘘更得分）
## 附录 A. 复现指南（从零到跑出主结果的命令序列）
## 附录 B. 排错日志精选（troubleshooting）
```

### 每周 checklist（贴在 journal 里打勾）

- [ ] 本周围绕里程碑有可演示的进展（哪怕是失败的 demo）
- [ ] 训练曲线截图入 journal；异常有假设与下一步
- [ ] 代码可跑：`uv run pytest && uv run ruff check .` 通过
- [ ] 周报 ≤200 字：进展 / 卡点 / 下周计划（learning_path 学习方法约定第 1 条）

---

## ✏️ 练习

1. **选题与计划**（★，约 45 分钟）
   选定一个题目（可以超出三个备选，但需自证工作量相当），按 W35–W38 填自己的里程碑表，
   每周验收物必须**可被他人检查**（文件/数字/录屏，不接受「基本完成」）。
   **交付物**：`journal/capstone_plan.md`。
2. **指标设计评审**（★★，约 40 分钟）
   为你的选题写完整指标表（≥4 个指标，每个含：定义、测量协议、目标线、优秀线），
   并指出其中哪个最容易被「刷分」（指标与真实能力脱节），给出防御措施。
   **交付物**：`journal/capstone_metrics.md`。
3. **算清评估预算**（★，约 20 分钟）
   用本 notebook 的 `wilson_ci` 回答：若你声称成功率 ≥75%，按 95% 置信度，
   试验次数 n 至少多少才能使 CI 下界 ≥0.75？（提示：写个循环找最小 n，假设观测成功率 0.85）
   **交付物**：`journal/eval_budget.py` + 结论一行。
4. **预写失败分析**（★★，约 30 分钟）
   项目还没开始，先写「最可能的 3 种失败方式」及每种的最小化验证实验
   （用 ≤1 天能做完的实验提前证伪）。这是 NASA 式 premortem。
   **交付物**：`journal/premortem.md`。

## 参考答案

<details>
<summary>练习 1：选题与计划（评审要点）</summary>

好计划的特征：① 每周验收物是名词（文件/数字/录屏）而非动词（「调试」「优化」）；
② W35 就有端到端最小 demo（哪怕性能很差）——把集成风险前置；
③ 明确写了「砍什么」：如视觉输入、真机部署列为加分项而非必需；
④ 依赖前置：需要 GPU/数据的步骤不排在等待资源到账的那周。
</details>

<details>
<summary>练习 2：指标设计评审（示例，以无人机穿越为例）</summary>

| 指标 | 测量协议 | 目标线 | 优秀线 |
|------|----------|--------|--------|
| 穿越成功率 | 固定 100 个 seed 的初始状态，5 门连续 | 60% | 85% |
| 平均用时 | 仅成功 trials，同 seed 集 | 30 s | 20 s |
| 能耗/轨迹平滑度 | 动作 L2 均值 | 基线+30% | 基线 |
| DR 鲁棒性 | 质量/风扰 ±20% 区间内成功率 | ≥50% | ≥75% |

最易刷分的是**穿越成功率**：若评估用训练同分布的 seed 集，过拟合初始状态即可虚高。
防御：评估 seed 集与训练 seed 集不相交 + 保留 10% 「隐藏」扰动维度只在评估时开。
</details>

<details>
<summary>练习 3：评估预算（参考代码与结论）</summary>

```python
import math

def wilson_ci(k, n, z=1.96):
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return center - margin, center + margin

for n in range(50, 2000, 10):
    k = math.ceil(0.85 * n)
    lo, _ = wilson_ci(k, n)
    if lo >= 0.75:
        print(f"最小 n = {n}（观测 {k}/{n}≈85% 时 CI 下界 {lo:.3f} ≥ 0.75）")
        break
```

结论：观测成功率 0.85 时约需 **n ≈ 190–200** 次试验，CI 下界才越过 0.75。
这就是「n=100 只是底线」的定量原因——指标越接近声明线，所需样本越多。
</details>

<details>
<summary>练习 4：premortem（示例，以抓取 IL 为例）</summary>

1. **数据质量不足**（脚本示教轨迹太单一）→ 最小实验：用 25 条数据训 BC，
   若成功率 <20% 且失败全是「差几厘米」，说明需要扰动增广，提前写增广管线；
2. **ACT 训练不收敛** → 最小实验：先过拟合 8 条数据（能否 100% 记住），
   不能则是 bug 而非数据问题，当天修；
3. **评估标准模糊**（「抓起」定义不清）→ 最小实验：第一天就写死成功判据代码
   （抬升 5 cm 持续 2 s 无掉落），并用手工场景验证判据无 false positive。
</details>

---

## 延伸阅读

- 算法与系统：[Isaac Lab](https://isaac-sim.github.io/IsaacLab/)、
  [MotrixLab](https://motrixlab.readthedocs.io/)、[ROS2 Jazzy](https://docs.ros.org/en/jazzy/)、[Nav2](https://docs.nav2.org/)
- 模仿学习论文：[Diffusion Policy (Chi et al., 2023)](https://arxiv.org/abs/2303.04137)、
  [ACT (Zhao et al., 2023)](https://arxiv.org/abs/2304.13705)、
  [RT-2 (Brohan et al., 2023)](https://arxiv.org/abs/2307.15818)
- 数据与训练工具：[LeRobot](https://github.com/huggingface/lerobot)、
  [diffusion_policy](https://github.com/real-stanford/diffusion_policy)
- 部署与并行：[rsl_rl](https://github.com/leggedrobotics/rsl_rl)、
  [Isaac Gym 论文](https://arxiv.org/abs/2108.10470)